# Open Darts - train the dart detector (free Colab GPU)

This notebook trains a small neural network that finds the **dart tips** and the **four calibration points** of the board
in a camera picture, at any camera angle. The app uses those four points to work out where the board is, so no manual
calibration is needed.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**.
2. Download `cropped_images.zip` (the 800x800 version) from https://ieee-dataport.org/open-access/deepdarts-dataset
   (free IEEE account needed) and put it in your Google Drive, in a folder called `darts`.
3. Run the cells in order. Training takes roughly 1-3 hours. The last cell downloads `dartnet.tflite`.
4. Copy `dartnet.tflite` to `app/src/main/assets/dartnet.tflite` in the project and rebuild the app.

In [ ]:
!pip -q install ultralytics pandas
!git clone https://github.com/Yaswanth1826/open-darts.git
!wget -q -O labels.pkl https://github.com/wmcnally/deep-darts/raw/master/dataset/labels.pkl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p dataset && unzip -q -o /content/drive/MyDrive/darts/cropped_images.zip -d dataset
!cp labels.pkl dataset/
!find dataset -maxdepth 3 -type d | head

If the folder listing above does not show `dataset/cropped_images/800/...`, move the folders so that path exists, then continue.

In [ ]:
!python open-darts/training/prepare_deepdarts.py --dataset-dir dataset --out yolo_darts

In [ ]:
!python open-darts/training/train.py --data yolo_darts/data.yaml --epochs 60 --imgsz 640 --batch 32

In [ ]:
from google.colab import files
files.download('dartnet.tflite')

**Own pictures:** the app's "Save every dart" switch stores frames in Pictures/OpenDarts. Label them (dart tips and the four
calibration points, boxes of 2.5% of the image) with a tool such as Roboflow or LabelImg in YOLO format, put them in
`images/` and `labels/` folders and pass `--extra your_folder` to `prepare_deepdarts.py`, then train again. Even a few
hundred pictures of your own board and room make the model noticeably better.